<a href="https://colab.research.google.com/github/ttktjmt/g1_spinkick_example/blob/main/notebooks/g1_spinkick_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🤖 G1 Spinkick Example Tutorial with MJLab**

### **📦 Setup and Installation**

In [1]:
# Clone the mjlab repository
!if [ ! -d "mjlab" ]; then git clone -q https://github.com/mujocolab/mjlab.git; fi

# Clone the g1_spinkick_example repository
!if [ ! -d "g1_spinkick_example" ]; then git clone -q https://github.com/ttktjmt/g1_spinkick_example.git; fi
%cd /content/g1_spinkick_example

# Install mjlab in editable mode
!uv add --editable ../mjlab -q

# Install g1_spinkick_example in editable mode
!uv pip install -e . -q

print("✓ Installation complete!")

/content/g1_spinkick_example
✓ Installation complete!


### **🔑 WandB Setup**

Add your WandB API key to Colab Secrets:
- `WANDB_API_KEY`: from [wandb.ai/authorize](https://wandb.ai/authorize)
- `WANDB_ENTITY`: your wandb entity name

In [2]:
import os

from google.colab import userdata

try:
  # Set this to disable wandb logger
  # os.environ['WANDB_MODE'] = 'disabled'

  # Set this to use wandb logger
  os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
  os.environ["WANDB_ENTITY"] = userdata.get("WANDB_ENTITY")

  print("✓ WandB configured successfully!")
except (AttributeError, KeyError):
  print("⚠ WandB secrets not found. Training will proceed without WandB logging.")

✓ WandB configured successfully!


### **Download assets and motion data**

In [3]:
SHAREPOINT_URL = "https://1sfu-my.sharepoint.com/:u:/g/personal/xbpeng_sfu_ca/EclKq9pwdOBAl-17SogfMW0Bved4sodZBQ_5eZCiz9O--w?e=bqXBaa&download=1"
OUT = "mimickit.zip"

!wget -O "$OUT" "$SHAREPOINT_URL"

--2026-01-10 09:05:49--  https://1sfu-my.sharepoint.com/:u:/g/personal/xbpeng_sfu_ca/EclKq9pwdOBAl-17SogfMW0Bved4sodZBQ_5eZCiz9O--w?e=bqXBaa&download=1
Resolving 1sfu-my.sharepoint.com (1sfu-my.sharepoint.com)... 13.107.136.10, 13.107.138.10, 2620:1ec:8f8::10, ...
Connecting to 1sfu-my.sharepoint.com (1sfu-my.sharepoint.com)|13.107.136.10|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: /personal/xbpeng_sfu_ca/Documents/MimicKit/MimicKit_Data.zip?ga=1 [following]
--2026-01-10 09:05:50--  https://1sfu-my.sharepoint.com/personal/xbpeng_sfu_ca/Documents/MimicKit/MimicKit_Data.zip?ga=1
Reusing existing connection to 1sfu-my.sharepoint.com:443.
HTTP request sent, awaiting response... 200 OK
Length: 216529818 (206M) [application/x-zip-compressed]
Saving to: ‘mimickit.zip’

mimickit.zip        100%[===================>] 206.50M  85.8MB/s    in 2.4s    

2026-01-10 09:05:53 (85.8 MB/s) - ‘mimickit.zip’ saved [216529818/216529818]



In [4]:
import os

# Create the data directory if it doesn't exist
DATA_DIR = "/content/g1_spinkick_example/data"
os.makedirs(DATA_DIR, exist_ok=True)

# Unzip the archive into the data directory
!unzip -o {OUT} -d {DATA_DIR}

print(f"✓ Archive '{OUT}' extracted to '{DATA_DIR}/' successfully!")

Archive:  mimickit.zip
   creating: /content/g1_spinkick_example/data/MimicKit_Data/
   creating: /content/g1_spinkick_example/data/MimicKit_Data/assets/
   creating: /content/g1_spinkick_example/data/MimicKit_Data/assets/g1/
  inflating: /content/g1_spinkick_example/data/MimicKit_Data/assets/g1/g1.usd  
  inflating: /content/g1_spinkick_example/data/MimicKit_Data/assets/g1/g1.xml  
  inflating: /content/g1_spinkick_example/data/MimicKit_Data/assets/g1/g1_mesh.xml  
  inflating: /content/g1_spinkick_example/data/MimicKit_Data/assets/g1/LICENSE.txt  
   creating: /content/g1_spinkick_example/data/MimicKit_Data/assets/g1/meshes/
  inflating: /content/g1_spinkick_example/data/MimicKit_Data/assets/g1/meshes/head_link.STL  
  inflating: /content/g1_spinkick_example/data/MimicKit_Data/assets/g1/meshes/left_ankle_pitch_link.STL  
  inflating: /content/g1_spinkick_example/data/MimicKit_Data/assets/g1/meshes/left_ankle_roll_link.STL  
  inflating: /content/g1_spinkick_example/data/MimicKit_Data

### **Convert**

Create a custom registry named "Motions" as described in the [mjlab README](https://github.com/mujocolab/mjlab?tab=readme-ov-file#2-motion-imitation)

Convert pkl to csv

In [5]:
!uv run pkl_to_csv.py \
    --pkl-file /content/g1_spinkick_example/data/MimicKit_Data/motions/g1/g1_spinkick.pkl \
    --csv-file g1_spinkick.csv \
    --duration 2.65 \
    --add-start-transition \
    --add-end-transition \
    --transition-duration 0.5 \
    --pad-duration 1.0

Installed 2 packages in 107ms
Loading /content/g1_spinkick_example/data/MimicKit_Data/motions/g1/g1_spinkick.pkl...
Loaded motion with shape: (78, 35)
FPS: 60
Original duration: 1.30 seconds (78 frames)
Repeating motion 3 times to reach 2.65s...
XY displacement per cycle: [-0.2881, 0.1935]
  (Forward motion detected - accumulating XY displacement only)
New motion shape: (159, 35)
New duration: 2.65 seconds (159 frames)

Adding start transition from safe standing pose to motion start...
Creating 0.5s start transition (30 frames)...
  First frame orientation: yaw=159.8°, pitch=5.1°, roll=-6.7°
  Start orientation: yaw=159.8° (matched), pitch=0°, roll=0°
After start transition, motion shape: (189, 35)
Total duration so far: 3.15 seconds

Adding end transition to safe standing pose (keeping yaw, resetting roll/pitch)...
Creating 0.5s end transition (30 frames)...
  Final frame orientation: yaw=164.9°, pitch=10.9°, roll=-5.7°
  Target orientation: yaw=164.9° (kept), pitch=0°, roll=0°
After 

Convert csv to npz

In [6]:
!MUJOCO_GL=egl uv run -m mjlab.scripts.csv_to_npz \
    --input-file g1_spinkick.csv \
    --output-name mimickit_spinkick_safe \
    --input-fps 60 \
    --output-fps 50 \
    --render

Warp 1.12.0.dev20260109 initialized:
   CUDA Toolkit 12.9, Driver 12.4
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.0.dev20260109
Warp DeprecationWarning: The namespace `warp.context` will soon be removed from the public API. It can still be accessed from `warp._src.context` but might be changed or removed without notice.
Warp DeprecationWarning: The symbol `warp.context.runtime` will soon be removed from the public API. It can still be accessed from `warp._src.context.runtime` but might be changed or removed without notice.
Module mujoco_warp._src.smooth e142b0f load on device 'cuda:0' took 6947.99 ms  (compiled)
Module mujoco_warp._src.collision_driver de6dc98 load on device 'cuda:0' took 151.61 ms  (compiled)
Module _nxn_broadphase__locals__kernel_86ad8c1b 86ad8c1 load on device 'cuda:0' took 565.45 ms  (compiled)
Module _primitive_narrowphase__locals__primitive_narrowphase_fd6a20a7 

## **Train**

In [ ]:
!MUJOCO_GL=egl CUDA_VISIBLE_DEVICES=0 uv run train \
    Mjlab-Spinkick-Unitree-G1 \
    --registry-name ${WANDB_ENTITY}/csv_to_npz/mimickit_spinkick_safe \
    --env.scene.num-envs 4096 \
    --agent.max-iterations 10_000

Streaming output truncated to the last 5000 lines.
 Metrics/motion/error_body_ang_vel: 4.3314
      Episode_Termination/time_out: 4.7083
    Episode_Termination/anchor_pos: 1.4167
    Episode_Termination/anchor_ori: 0.0000
   Episode_Termination/ee_body_pos: 7.6667
Episode_Termination/base_ang_vel_exceed: 1.2500
--------------------------------------------------------------------------------
                   Total timesteps: 54657024
                    Iteration time: 5.29s
                      Time elapsed: 00:47:18
                               ETA: 03:34:22

################################################################################
                      Learning iteration 556/20000                      

                       Computation: 18876 steps/s (collection: 4.634s, learning 0.574s)
             Mean action noise std: 0.43
          Mean value_function loss: 0.1625
               Mean surrogate loss: -0.0074
                 Mean entropy loss: 16.5014
            

### **🌐 Launch Viser API**

In [ ]:
import subprocess
import sys
import re

run_id = "6fo2umjl"  # Update with your run-id

process = subprocess.Popen(
  [
    "uv",
    "run",
    "play",
    "Mjlab-Spinkick-Unitree-G1",
    "--wandb-run-path",
    f"{os.environ["WANDB_ENTITY"]}/csv_to_npz/{run_id}",
    "--num_envs",
    "8",
  ],
  stdout=subprocess.PIPE,
  stderr=subprocess.STDOUT,
  universal_newlines=True,
  bufsize=1,
)

detected_port = None

for line in process.stdout:
  print(line, end="")
  sys.stdout.flush()

  # Extract port number from viser output
  port_match = re.search(r":(\d{4})", line)
  if port_match and "viser" in line.lower():
    detected_port = int(port_match.group(1))
    print("\n" + "=" * 34)
    print(f"✅ Server is running on port {detected_port}!")
    print("=" * 34)
    break

### **🖥️ Embed Client as iframe**

In [ ]:
from google.colab import output

port = detected_port if detected_port else 8081
output.serve_kernel_port_as_iframe(port=port, height=700)